# تبدیل موازی کتاب‌ها (نسخه‌ی 23)

کتاب‌های بلند (به‌خصوص اسکن‌شده‌ها که OCR می‌خواهند) ساعت‌ها طول می‌کشند. این نوت‌بوک هر کتاب را به بخش‌های
۱۰ صفحه‌ای می‌شکند و **هر بخش همین که تمام شد در Drive ذخیره می‌شود**؛ اگر قطع شد، دوباره اجرا کنید و از بخش
بعدی ادامه می‌دهد.

**موازی:** همین نوت‌بوک را در 3 تب جدا باز کنید و در هر تب `WORKER` را ۱، ۲ و ۳ بگذارید؛ هر تب
سهم خودش از بخش‌های همه‌ی کتاب‌ها را تبدیل می‌کند. اگر کولب همه را اجازه نداد، هر تعداد که شد کافی است: تب‌ها
بخش‌های ناتمام هم را با اجرای دوباره با `WORKERS = 1` تمام می‌کنند.

بعد از تمام شدن، نوت‌بوک «افزودن مقاله‌های تازه» را اجرا کنید: بخش‌ها را به هم وصل، نمایه و روی سایت می‌فرستد.

**با حساب دیگر (مثلاً دوستی که GPU دارد):** دسترسی Editor به پوشه‌ی `RAG-Optics` لازم است (کتاب‌ها در
`RAG-Optics/_books` هستند). او در Drive روی پوشه راست‌کلیک کند ← Organize ← Add shortcut ← My Drive، بعد این نوت‌بوک
را با T4 GPU اجرا کند. بخش‌ها مستقیم در `RAG-Optics/_RAG_processed` ذخیره می‌شوند.

In [ ]:
#@title ▶️ تبدیل کتاب‌ها
WORKER = 1  #@param {type:"integer"}
WORKERS = 1  #@param {type:"integer"}  — 3 tabs in parallel: WORKERS = 3 and WORKER = 1, 2, 3
COLLECTION = "/content/drive/MyDrive/RAG-Optics"  #@param {type:"string"}
# the books wait in RAG-Optics/_books (folders starting with _ are not papers of the collection); the finished parts
# go straight to the collection's conversion cache, RAG-Optics/_RAG_processed, where the new-papers run finds them
DOWNLOADS = COLLECTION + '/_books'

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')
# The per-paper lists (OCR, equation decoding, wrong PDFs, reconversions, papers whose site points are made
# elsewhere) are read from the data folder on Drive; the repository's copies are the current ones.
import glob, os, shutil
if 'WORK' in globals():
    os.makedirs(f'{WORK}/data/rag-optics', exist_ok=True)
    for f in glob.glob('/content/RAG/data/rag-optics/*.txt'):
        shutil.copy(f, f'{WORK}/data/rag-optics/')

!pip install -q "rapidocr>=3" onnxruntime 2>&1 | grep -iE "^error" || true
if not os.path.isdir(DOWNLOADS):
    raise SystemExit(f'{DOWNLOADS} not found. If RAG-Optics was shared with you: in Google Drive, right-click it → '
                     'Organize → Add shortcut → My Drive, then run this cell again.')
os.environ['MATRAG_DATA_DIR'] = '/content/book_data'  # only the link to the parts folder lives here
os.environ['MATRAG_CORPUS'] = 'rag-optics'
!matrag link "{COLLECTION}" > /dev/null
# the handbooks of the new-papers notebook: file name -> doc_id (its DOI, or ISBN, as a file name)
BOOKS = {'ghosh1998.pdf': 'isbn_9780122818554', 'bass2001.pdf': '10.1036_0071414797',
         'weber2003.pdf': '10.1201_9781420050196', 'weber1986.pdf': 'isbn_9780849335044'}
import pathlib
for name, doc_id in BOOKS.items():
    pdf = pathlib.Path(DOWNLOADS) / name
    if not pdf.exists():
        print('not in the downloads folder, skipped:', name)
        continue
    !matrag convert-book "{pdf}" {doc_id} --shard {WORKER - 1} --shards {WORKERS} 2>&1 | grep --line-buffered -vE "Warning|it/s\]"
print(f'✅ Share {WORKER} of {WORKERS} finished.')